In [ ]:
"""Utilities: data I/O, padding helpers, and Data_Batch classes."""
# Utils
import argparse
import datetime
import os
import pickle
import sys
import time
from collections import defaultdict
from math import sqrt

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import mean_squared_error
from torch.utils.data import DataLoader
from tqdm import tqdm

import test

def open_pkl_file2(path, description):
    """Load a pickle file and return incremental prefixes of sequences.

    Args:
        path (str): Path to the .pkl file.
        description (str): Key inside the pickled object to use.

    Returns:
        tuple: (time_durations, type_seqs) where each is a list of incremental prefixes.
    """
    with open(path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')
        data = data[description]
    time_durations = []
    type_seqs = []
    seq_lens = []
    for i in range(len(data)):
        end = 2
        while end <= len(data[i]):
            seq_lens.append(end)
            type_seqs.append([[int(event['type_event']) for event in data[i][:end]]])
            time_durations.append([[float(event['time_since_last_event']) for event in data[i][:end]]])
            end += 1
    return time_durations, type_seqs

def open_pkl_file(path, description):
    """Load a pickle file and return full sequences as tensors.

    Args:
        path (str): Path to the .pkl file.
        description (str): Key inside the pickled object to use.

    Returns:
        tuple: (time_durations, type_seqs, seq_lens) where tensors are returned.
    """
    with open(path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')
        data = data[description]
    time_durations = []
    type_seqs = []
    seq_lens = []
    for i in range(len(data)):
        seq_lens.append(len(data[i]))
        type_seqs.append(torch.LongTensor([int(event['type_event']) for event in data[i]]))
        time_durations.append(torch.FloatTensor([float(event['time_since_last_event']) for event in data[i]]))
    return time_durations, type_seqs, seq_lens

def open_txt_file(path):
    """Read a whitespace-separated text file of timestamps and return inter-arrival times.

    Args:
        path (str): Path to the text file.

    Returns:
        tuple: (time_duration, seq_lens_list) where each entry is a tensor and sequence length.
    """
    f = open(path, 'r')
    data_file = f.readlines()
    f.close()
    time_duration = []
    seq_lens_list = []
    # total_time_list = []
    for line in data_file:
        data = line.split(" ")
        a_list = []
        previous = 0
        lens = 0
        for i in range(len(data)):
            if data[i] != "\n":
                a_list.append(float(data[i]) - previous)
                previous = float(data[i])
                lens += 1
        time_duration.append(torch.tensor(a_list))
        # total_time_list.append(previous)
        seq_lens_list.append(lens)
    return time_duration, seq_lens_list

def get_index_txt(duration):
    """Create a zero tensor for event types matching duration shapes.

    Args:
        duration (list[tensor]): list of duration tensors.

    Returns:
        torch.Tensor: stacked zero tensors (long dtype) matching durations.
    """
    # Sets to all 0s
    type_list = []
    for i in range(len(duration)):
        a_list = torch.zeros(size=duration[i].shape, dtype=torch.long)
        type_list.append(a_list)
    type_list = torch.stack(type_list)
    return type_list

def padding_full(time_duration, type_train, seq_lens_list, type_size):
    """Pad variable-length sequences to (batch, max_len+1).

    Args:
        time_duration (list[tensor]): list of duration tensors.
        type_train (list[tensor]): list of type tensors.
        seq_lens_list (list[int]): list of sequence lengths.
        type_size (int): padding index for types.

    Returns:
        tuple: (time_duration_padded, type_train_padded).
    """
    max_len = max(seq_lens_list)
    batch_size = len(time_duration)
    time_duration_padded = torch.zeros(size=(batch_size, max_len+1))
    type_train_padded = torch.zeros(size=(batch_size, max_len+1), dtype=torch.long)
    for idx in range(batch_size):
        time_duration_padded[idx, 1:seq_lens_list[idx]+1] = time_duration[idx]
        type_train_padded[idx, 0] = type_size
        type_train_padded[idx, 1:seq_lens_list[idx]+1] = type_train[idx]
    return time_duration_padded, type_train_padded

def padding_full_feats(time_duration, type_train, seq_lens_list, type_size, event_features):
    """Pad durations, types, and event features to uniform length.

    Returns padded (time_duration, type_train, feat_train) tensors.
    """
    max_len = max(seq_lens_list)
    batch_size = len(time_duration)
    feat_dim = event_features[0].shape[-1]

    time_duration_padded = torch.zeros(size=(batch_size, max_len+1))
    type_train_padded = torch.zeros(size=(batch_size, max_len+1), dtype=torch.long)
    feat_train_padded = torch.zeros(size=(batch_size, max_len+1, feat_dim))
    for idx in range(batch_size):
        time_duration_padded[idx, 1:seq_lens_list[idx]+1] = time_duration[idx]
        type_train_padded[idx, 0] = type_size
        type_train_padded[idx, 1:seq_lens_list[idx]+1] = type_train[idx]
        feat_train_padded[idx, 1:seq_lens_list[idx]+1]=event_features[idx]

    return time_duration_padded, type_train_padded, feat_train_padded

def padding_seq_len(duration, types, type_size, seq_len):
    """Create sliding windows of fixed `seq_len` from variable-length sequences.

    Args:
        duration (list[tensor]): list of duration tensors.
        types (list[tensor]): list of type tensors.
        type_size (int): padding type index.
        seq_len (int): target window length.

    Returns:
        tuple: (time_duration, type_lists, seq_lens_list).
    """
    time_duration = []
    type_lists = []
    seq_lens_list = []
    batch_size = len(duration)
    for i in range(batch_size):
        end = seq_len
        while end <= duration[i].shape.__getitem__(-1):
            start = end - seq_len
            duration_list = [0]
            type_list = [type_size]
            duration_list = duration_list + duration[i][start:end].tolist()
            type_list = type_list + types[i][start:end].tolist()
            time_duration.append(duration_list)
            type_lists.append(type_list)
            seq_lens_list.append(seq_len)
            end += 1
    time_duration = torch.tensor(time_duration)
    type_lists = torch.tensor(type_lists)
    return time_duration, type_lists, seq_lens_list

def padding_seq_len_feats(duration, types, type_size, seq_len, event_features):
    """Create sliding windows and include feature padding for each window.

    Returns padded durations, types, seq lengths, and event feature tensors.
    """
    time_duration = []
    type_lists = []
    seq_lens_list = []
    event_feat_padded=[]
    batch_size = len(duration)
    featsize = len(event_features[0][0])
    for i in range(batch_size):
        end = seq_len
        while end <= duration[i].shape.__getitem__(-1):
            start = end - seq_len
            duration_list = [0]
            featlist = [[0]*featsize]
            type_list = [type_size]
            duration_list = duration_list + duration[i][start:end].tolist()
            type_list = type_list + types[i][start:end].tolist()
            featlist = featlist + event_features[i][start:end].tolist()

            time_duration.append(duration_list)
            type_lists.append(type_list)
            seq_lens_list.append(seq_len)
            event_feat_padded.append(featlist)
            end += 1

    time_duration = torch.tensor(time_duration)
    type_lists = torch.tensor(type_lists)
    event_feat_padded = torch.tensor(event_feat_padded)
    return time_duration, type_lists, seq_lens_list, event_feat_padded

def generate_simulation(durations, seq_len):
    """Generate Monte-Carlo simulated timestamps inside each sequence interval.

    Args:
        durations (torch.Tensor): tensor shape (batch, max_seq) of inter-arrival durations.
        seq_len (list[int]): list of effective sequence lengths for each batch item.

    Returns:
        tuple: (sim_durations, total_time_seqs, sim_duration_index).
    """
    max_seq_len = max(seq_len)
    simulated_len = max_seq_len * 5
    sim_durations = torch.zeros(durations.shape[0], simulated_len)
    sim_duration_index = torch.zeros(durations.shape[0], simulated_len, dtype=torch.long)
    total_time_seqs = []
    for idx in range(durations.shape[0]):
        time_seq = durations[idx, :seq_len[idx]].cumsum(dim=0)
        total_time = time_seq[-1].item()
        total_time_seqs.append(total_time)
        sim_time_seq, _ = torch.sort(torch.empty(simulated_len).uniform_(0, total_time))
        sim_duration = torch.zeros(simulated_len)

        for idx2 in range(time_seq.shape.__getitem__(-1)):
            duration_index = sim_time_seq > time_seq[idx2].item()
            sim_duration[duration_index] = sim_time_seq[duration_index] - time_seq[idx2]
            sim_duration_index[idx][duration_index] = idx2

        sim_durations[idx, :] = sim_duration[:]
    total_time_seqs = torch.tensor(total_time_seqs)
    return sim_durations, total_time_seqs, sim_duration_index

class Data_Batch:
    """Simple dataset wrapper providing dictionary samples for DataLoader.

    Args:
        duration (torch.Tensor): duration tensor.
        events (torch.Tensor): event type tensor.
        seq_len (list[int]): sequence lengths.
    """
    def __init__(self, duration, events, seq_len):
        self.duration = duration
        self.events = events
        self.seq_len = seq_len

    def __len__(self):
        """Return number of samples.

        Returns:
            int: number of samples in dataset.
        """
        return self.events.shape[0]

    def __getitem__(self, index):
        """Return a dictionary sample for the given index.

        Args:
            index (int): sample index.

        Returns:
            dict: {'event_seq', 'duration_seq', 'seq_len'}.
        """
        sample = {
            'event_seq': self.events[index],
            'duration_seq': self.duration[index],
            'seq_len': self.seq_len[index]
        }
        return sample

class Data_Batch_Feat:
    """Dataset wrapper that includes extra event features.

    Args:
        duration (torch.Tensor): duration tensor.
        events (torch.Tensor): event type tensor.
        seq_len (list[int]): sequence lengths.
        features (torch.Tensor): per-event features.
    """
    def __init__(self, duration, events, seq_len, features):
        self.duration = duration
        self.events = events
        self.seq_len = seq_len
        self.features = features  # new parameter

    def __len__(self):
        """Return number of samples.

        Returns:
            int: number of samples in dataset.
        """
        return self.events.shape[0]

    def __getitem__(self, index):
        """Return a dictionary sample including features for the given index.

        Args:
            index (int): sample index.

        Returns:
            dict: {'event_seq', 'duration_seq', 'seq_len', 'features'}.
        """
        sample = {
            'event_seq': self.events[index],
            'duration_seq': self.duration[index],
            'seq_len': self.seq_len[index],
            'features': self.features[index]  # include features in each sample
        }
        return sample

In [ ]:
"""Continuous-time LSTM (CTLSTM) cell implementation."""
# conttimecell.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class CTLSTMCell(nn.Module):
    """Continuous-time LSTM cell used by the conttime models.

    Args:
        hidden_dim (int): hidden dimensionality.
        beta (float): softplus beta used for decay parameter.
        device (str|None): device string (optional).
    """
    def __init__(self, hidden_dim, beta=1.0, device=None):
        super(CTLSTMCell, self).__init__()

        device = device or 'cpu'
        self.device = torch.device(device)

        self.hidden_dim = hidden_dim

        self.linear = nn.Linear(hidden_dim * 2, hidden_dim * 7, bias=True)
        self.beta = beta

    def forward(
            self, rnn_input,
            hidden_t_i_minus, cell_t_i_minus, cell_bar_im1):

        """Compute the CTLSTM cell update for one timestep.

        Args:
            rnn_input (torch.Tensor): input vector for current timestep.
            hidden_t_i_minus (torch.Tensor): previous hidden state.
            cell_t_i_minus (torch.Tensor): previous cell state.
            cell_bar_im1 (torch.Tensor): previous cell_bar state.

        Returns:
            tuple: (cell_i, cell_bar_i, gate_decay, gate_output).
        """

        dim_of_hidden = rnn_input.dim() - 1

        input_i = torch.cat((rnn_input, hidden_t_i_minus), dim=dim_of_hidden)
        output_i = self.linear(input_i)

        gate_input, \
        gate_forget, gate_output, gate_pre_c, \
        gate_input_bar, gate_forget_bar, gate_decay = output_i.chunk(
            7, dim_of_hidden)

        gate_input = torch.sigmoid(gate_input)
        gate_forget = torch.sigmoid(gate_forget)
        gate_output = torch.sigmoid(gate_output)
        gate_pre_c = torch.tanh(gate_pre_c)
        gate_input_bar = torch.sigmoid(gate_input_bar)
        gate_forget_bar = torch.sigmoid(gate_forget_bar)
        gate_decay = F.softplus(gate_decay, beta=self.beta)

        cell_i = gate_forget * cell_t_i_minus + gate_input * gate_pre_c
        cell_bar_i = gate_forget_bar * cell_bar_im1 + gate_input_bar * gate_pre_c

        return cell_i, cell_bar_i, gate_decay, gate_output

    def decay(self, cell_i, cell_bar_i, gate_decay, gate_output, dtime):
        """Decay cell states forward by time interval `dtime`.

        Args:
            cell_i (torch.Tensor): current cell.
            cell_bar_i (torch.Tensor): baseline cell.
            gate_decay (torch.Tensor): decay rate per dim.
            gate_output (torch.Tensor): output gate.
            dtime (torch.Tensor): time elapsed to next event.

        Returns:
            tuple: (cell_t_ip1_minus, hidden_t_ip1_minus).
        """
        if dtime.dim() < cell_i.dim():
            dtime = dtime.unsqueeze(cell_i.dim()-1).expand_as(cell_i)

        cell_t_ip1_minus = cell_bar_i + (cell_i - cell_bar_i) * torch.exp(
            -gate_decay * dtime)
        hidden_t_ip1_minus = gate_output * torch.tanh(cell_t_ip1_minus)

        return cell_t_ip1_minus, hidden_t_ip1_minus

In [ ]:
# conttimefeat
import torch
# import cont_time_cell
import torch.nn as nn
import torch.nn.functional as F

class FeatureAttention(nn.Module):
    """Apply feature-wise attention across the time dimension.

    Args:
        time_steps (int): Number of time steps in the input sequence.
        features (int): Number of feature channels per time step.
    """

    def __init__(self, time_steps, features):
        """Initialize the attention layer.

        Args:
            time_steps (int): Sequence length for the time dimension.
            features (int): Number of feature channels.
        """
        super(FeatureAttention, self).__init__()
        self.time_steps = time_steps
        self.features = features
        self.attention = nn.Linear(time_steps, time_steps)

    def forward(self, x):
        """Apply attention weighting to each feature independently.

        Args:
            x (torch.Tensor): Input tensor of shape ``(batch, time_steps, features)``.

        Returns:
            torch.Tensor: Attention-weighted tensor of the same shape.
        """
        x_perm = x.permute(0, 2, 1)
        attn_scores = torch.tanh(self.attention(x_perm))
        attn_weights = F.softmax(attn_scores, dim=-1)
        attended = attn_weights * x_perm
        attended = attended.permute(0, 2, 1)
        return attended


class CNNAttentionLSTM(nn.Module):
    """Compact feature encoder using attention, 1D convolution, and LSTM."""

    def __init__(self, time_steps, features):
        """Initialize the CNN-attention-LSTM encoder.

        Args:
            time_steps (int): Number of time steps in the input sequence.
            features (int): Number of input features per time step.
        """
        super(CNNAttentionLSTM, self).__init__()
        self.time_steps = time_steps
        self.features = features
        self.attention = FeatureAttention(time_steps, features)

        self.conv1d = nn.Conv1d(in_channels=features, out_channels=32, kernel_size=5)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.lstm = nn.LSTM(
            input_size=16,
            hidden_size=40,
            num_layers=1,
            batch_first=True,
            dropout=0.3
        )
        self.fc1 = nn.Linear(40, 40)
        self.fc2 = nn.Linear(40, 1)
    def forward(self, x):
        x = self.attention(x)

        x = x.permute(0, 2, 1)
        x = self.conv1d(x)
        x = x.permute(0, 2, 1)
        
        x = F.leaky_relu(x)
        x = self.pool1(x)
        x, _ = self.lstm(x)
        x = F.leaky_relu(self.fc1(x[:, -1]))
        x = F.sigmoid(self.fc2(x))
        return x
    
class ConttimeFeat(nn.Module):
    def __init__(self, n_types, beta=0.1, hid_dim=32, extra_dim=0, lr=0.01):
        super().__init__()
        self.n_types = n_types
        self.beta = beta
        self.hid_dim = hid_dim
        self.extra_dim = extra_dim

        self.emb = nn.Embedding(self.n_types + 1, self.hid_dim)
        self.input_dim = self.hid_dim + (self.extra_dim if self.extra_dim > 0 else 0)

        # self.extra_emb = nn.Linear(extra_dim, hid_dim) if extra_dim > 0 else None
        self.lstm_cell = CTLSTMCell(self.input_dim, beta)
        self.hidden_lambda = nn.Linear(self.input_dim, self.n_types)
        self.optimizer = torch.optim.Adam(self.parameters(), lr=lr)

    def forward(self, types, dtime, extra_feats):
        numb_seq, seq_len = dtime.shape
        self.hid_layer_minus = torch.zeros(numb_seq, self.input_dim)
        self.cell_minus = torch.zeros(numb_seq, self.input_dim)
        self.cell_bar_minus = torch.zeros(numb_seq, self.input_dim)

        h_list, c_list, c_bar_list, decay_list, gate_out_list = [], [], [], [], []

        for i in range(seq_len - 1):
            type_input = self.emb(types[:, i])
            if extra_feats is not None:
                input_combined = torch.cat([type_input, extra_feats[:, i]], dim=-1)
            else:
                input_combined = type_input

            cell_i, cell_bar_updated, gate_decay, gate_output = self.lstm_cell(
                input_combined, self.hid_layer_minus, self.cell_minus, self.cell_bar_minus
            )
            self.cell_minus, self.hid_layer_minus = self.lstm_cell.decay(
                cell_i, cell_bar_updated, gate_decay, gate_output, dtime[:, i+1]
            )

            h_list.append(self.hid_layer_minus)
            c_list.append(cell_i)
            c_bar_list.append(cell_bar_updated)
            decay_list.append(gate_decay)
            gate_out_list.append(gate_output)

        return (
            torch.stack(h_list),
            torch.stack(c_list),
            torch.stack(c_bar_list),
            torch.stack(decay_list),
            torch.stack(gate_out_list),
        )


    def train_batch(self, batch, sim_dur, total_time_lists, seq_len_lists, time_simulation_index):
        if len(batch)==2:
            types, dtime = batch
            extra_feats=None
        else:
            types, dtime, extra_feats = batch

        h_out, c_out, c_bar_out, decay_out, gate_out = self.forward(types, dtime, extra_feats=extra_feats)
        part_one_likelihood, part_two_likelihood, sum_likelihood = self.conttime_loss(h_out, c_out, c_bar_out,
                                                                                      decay_out, gate_out,
                                                                                      types, sim_dur, total_time_lists,
                                                                                      seq_len_lists, time_simulation_index)
        loss = -(torch.sum(part_one_likelihood - part_two_likelihood))
        loss.backward()
        self.optimizer.step()
        self.optimizer.zero_grad()
        return loss

    def conttime_loss(self, h, c, c_bar, decay, o, event_seqs, sim_duration, total_time_list, seq_len_lists,
                      time_simulation_index):
        # print(c)
        batch_size = event_seqs.shape[0]
        # print(time_simulation_index)
        sim_len = time_simulation_index.shape[1]
        part_one_likelihood = torch.zeros(batch_size)
        sum_likelihood = torch.zeros(batch_size)

        # Get lambda
        type_intensity = torch.nn.functional.softplus(self.hidden_lambda(h)).transpose(0,1)
        for idx in range(batch_size):
            event_seq = event_seqs[idx]
            seq_len = seq_len_lists[idx]
            part_one_likelihood[idx] = torch.sum(torch.log(type_intensity[idx, torch.arange(seq_len), event_seq[1:seq_len+1]]))
            sum_likelihood[idx] = torch.sum(torch.log(torch.sum(type_intensity[idx, torch.arange(seq_len), :], dim=-1)))

        c_sim = []
        c_bar_sim = []
        decay_sim = []
        o_sim = []
        for j in range(batch_size):
            layer_c = c[time_simulation_index[j], j, :]
            c_sim.append(layer_c)
            layer_c_bar = c_bar[time_simulation_index[j], j, :]
            c_bar_sim.append(layer_c_bar)
            layer_decay = decay[time_simulation_index[j], j, :]
            decay_sim.append(layer_decay)
            layer_o = o[time_simulation_index[j], j, :]
            o_sim.append(layer_o)
        c_sim = torch.stack(c_sim).transpose(0,1)
        c_bar_sim = torch.stack(c_bar_sim).transpose(0,1)
        decay_sim = torch.stack(decay_sim).transpose(0,1)
        o_sim = torch.stack(o_sim).transpose(0,1)
        h_sim_list = []
        # print(c_sim)
        for idx in range(sim_duration.shape[1]):
            cell_next, h_sim = self.lstm_cell.decay(c_sim[idx], c_bar_sim[idx], decay_sim[idx], o_sim[idx], sim_duration[:, idx])
            h_sim_list.append(h_sim)
        h_sim_list = torch.stack(h_sim_list)
        sim_intensity = torch.nn.functional.softplus(self.hidden_lambda(h_sim_list)).transpose(0,1)
        part_two_likelihood = torch.zeros(batch_size)
        for idx in range(batch_size):
            coefficient = total_time_list[idx] / sim_len
            part_two_likelihood[idx] = torch.sum(torch.sum(sim_intensity[idx, torch.arange(sim_len), :])) * coefficient


        return part_one_likelihood, part_two_likelihood, sum_likelihood
    
class ConttimeFeatAttnLSTM(nn.Module):
    def __init__(self, n_types, seq_len, beta=0.1, hid_dim=32, extra_dim=0, lr=0.01):
        super().__init__()
        self.n_types = n_types
        self.beta = beta
        self.hid_dim = hid_dim
        self.extra_dim = extra_dim
        
        self.emb = nn.Embedding(self.n_types + 1, self.hid_dim)
        self.input_dim = self.hid_dim + 2*(self.extra_dim if self.extra_dim > 0 else 0)

        self.attention = FeatureAttention(seq_len, self.extra_dim)

        # self.extra_emb = nn.Linear(extra_dim, hid_dim) if extra_dim > 0 else None
        self.lstm_cell = CTLSTMCell(self.input_dim, beta)
        self.hidden_lambda = nn.Linear(self.input_dim, self.n_types)
        self.optimizer = torch.optim.Adam(self.parameters(), lr=lr)

    def forward(self, types, dtime, extra_feats):
        numb_seq, seq_len = dtime.shape
        self.hid_layer_minus = torch.zeros(numb_seq, self.input_dim)
        self.cell_minus = torch.zeros(numb_seq, self.input_dim)
        self.cell_bar_minus = torch.zeros(numb_seq, self.input_dim)

        h_list, c_list, c_bar_list, decay_list, gate_out_list = [], [], [], [], []

        temp = self.attention(extra_feats)
        extra_feats = torch.cat([extra_feats, temp], dim=-1)
        
        for i in range(seq_len - 1):
            type_input = self.emb(types[:, i])
            if extra_feats is not None:
                input_combined = torch.cat([type_input, extra_feats[:, i]], dim=-1)
            else:
                input_combined = type_input
                
            cell_i, cell_bar_updated, gate_decay, gate_output = self.lstm_cell(
                input_combined, self.hid_layer_minus, self.cell_minus, self.cell_bar_minus
            )
            self.cell_minus, self.hid_layer_minus = self.lstm_cell.decay(
                cell_i, cell_bar_updated, gate_decay, gate_output, dtime[:, i+1]
            )

            h_list.append(self.hid_layer_minus)
            c_list.append(cell_i)
            c_bar_list.append(cell_bar_updated)
            decay_list.append(gate_decay)
            gate_out_list.append(gate_output)

        return (
            torch.stack(h_list),
            torch.stack(c_list),
            torch.stack(c_bar_list),
            torch.stack(decay_list),
            torch.stack(gate_out_list),
        )


    def train_batch(self, batch, sim_dur, total_time_lists, seq_len_lists, time_simulation_index):
        if len(batch)==2:
            types, dtime = batch
            extra_feats=None
        else:
            types, dtime, extra_feats = batch

        h_out, c_out, c_bar_out, decay_out, gate_out = self.forward(types, dtime, extra_feats=extra_feats)
        part_one_likelihood, part_two_likelihood, sum_likelihood = self.conttime_loss(h_out, c_out, c_bar_out,
                                                                                      decay_out, gate_out,
                                                                                      types, sim_dur, total_time_lists,
                                                                                      seq_len_lists, time_simulation_index)
        loss = -(torch.sum(part_one_likelihood - part_two_likelihood))
        loss.backward()
        self.optimizer.step()
        self.optimizer.zero_grad()
        return loss

    def conttime_loss(self, h, c, c_bar, decay, o, event_seqs, sim_duration, total_time_list, seq_len_lists,
                      time_simulation_index):
        # print(c)
        batch_size = event_seqs.shape[0]
        # print(time_simulation_index)
        sim_len = time_simulation_index.shape[1]
        part_one_likelihood = torch.zeros(batch_size)
        sum_likelihood = torch.zeros(batch_size)

        # Get lambda
        type_intensity = torch.nn.functional.softplus(self.hidden_lambda(h)).transpose(0,1)
        for idx in range(batch_size):
            event_seq = event_seqs[idx]
            seq_len = seq_len_lists[idx]
            part_one_likelihood[idx] = torch.sum(torch.log(type_intensity[idx, torch.arange(seq_len), event_seq[1:seq_len+1]]))
            sum_likelihood[idx] = torch.sum(torch.log(torch.sum(type_intensity[idx, torch.arange(seq_len), :], dim=-1)))

        c_sim = []
        c_bar_sim = []
        decay_sim = []
        o_sim = []
        for j in range(batch_size):
            layer_c = c[time_simulation_index[j], j, :]
            c_sim.append(layer_c)
            layer_c_bar = c_bar[time_simulation_index[j], j, :]
            c_bar_sim.append(layer_c_bar)
            layer_decay = decay[time_simulation_index[j], j, :]
            decay_sim.append(layer_decay)
            layer_o = o[time_simulation_index[j], j, :]
            o_sim.append(layer_o)
        c_sim = torch.stack(c_sim).transpose(0,1)
        c_bar_sim = torch.stack(c_bar_sim).transpose(0,1)
        decay_sim = torch.stack(decay_sim).transpose(0,1)
        o_sim = torch.stack(o_sim).transpose(0,1)
        h_sim_list = []
        # print(c_sim)
        for idx in range(sim_duration.shape[1]):
            cell_next, h_sim = self.lstm_cell.decay(c_sim[idx], c_bar_sim[idx], decay_sim[idx], o_sim[idx], sim_duration[:, idx])
            h_sim_list.append(h_sim)
        h_sim_list = torch.stack(h_sim_list)
        sim_intensity = torch.nn.functional.softplus(self.hidden_lambda(h_sim_list)).transpose(0,1)
        part_two_likelihood = torch.zeros(batch_size)
        for idx in range(batch_size):
            coefficient = total_time_list[idx] / sim_len
            part_two_likelihood[idx] = torch.sum(torch.sum(sim_intensity[idx, torch.arange(sim_len), :])) * coefficient


        return part_one_likelihood, part_two_likelihood, sum_likelihood

class Conttime(nn.Module):
    def __init__(self, n_types, beta = 0.1, hid_dim=32, lr=0.01):
        # n_types is number of marks
        # Model is as follows: input: as the marks history for some time steps and dtimes of the history
        # Output is: h_out, c_out, and other gate features
        self.n_types = n_types
        self.beta = beta
        self.hid_dim = hid_dim

        super(Conttime, self).__init__()
        self.emb = nn.Embedding(self.n_types+1, self.hid_dim)
        
        self.lstm_cell = CTLSTMCell(hid_dim, beta)
        # Input is the hidden state after getting embedding, previous states (h, c, cbar)
        # Output is next (h, c, cbar)

        self.hidden_lambda = nn.Linear(self.hid_dim, self.n_types)
        self.optimizer = torch.optim.Adam(self.parameters(), lr=lr)

    def train_batch(self, batch, sim_dur, total_time_lists, seq_len_lists, time_simulation_index):
        types, dtime = batch
        h_out, c_out, c_bar_out, decay_out, gate_out = self.forward(types, dtime)
        part_one_likelihood, part_two_likelihood, sum_likelihood = self.conttime_loss(h_out, c_out, c_bar_out,
                                                                                      decay_out, gate_out,
                                                                                      types, sim_dur, total_time_lists,
                                                                                      seq_len_lists, time_simulation_index)
        loss = -(torch.sum(part_one_likelihood - part_two_likelihood))
        loss.backward()
        self.optimizer.step()
        self.optimizer.zero_grad()
        return loss

    def forward(self, types, dtime):
        numb_seq, seq_len = dtime.shape

        # Initially the lstm states are 0
        self.hid_layer_minus = torch.zeros(numb_seq, self.hid_dim, dtype=torch.float32)
        self.cell_minus = torch.zeros(numb_seq, self.hid_dim, dtype=torch.float32)
        self.cell_bar_minus = torch.zeros(numb_seq, self.hid_dim, dtype=torch.float32)

        # The output is the h, c, cbar, decay, gate_out at all time steps
        h_list, c_list, c_bar_list, decay_list, gate_out_list = [],[],[],[],[]
        for i in range(seq_len-1):
            type_input = self.emb(types[:,i])
            cell_i, cell_bar_updated, gate_decay, gate_output = self.lstm_cell(type_input, self.hid_layer_minus, self.cell_minus, self.cell_bar_minus)
            self.cell_minus, self.hid_layer_minus = self.lstm_cell.decay(cell_i, cell_bar_updated, gate_decay, gate_output, dtime[:,i+1])

            h_list.append(self.hid_layer_minus)
            c_list.append(cell_i)
            c_bar_list.append(cell_bar_updated)
            decay_list.append(gate_decay)
            gate_out_list.append(gate_output)
        h_out = torch.stack(h_list)
        c_out = torch.stack(c_list)
        c_bar_out = torch.stack(c_bar_list)
        decay_out = torch.stack(decay_list)
        gate_out = torch.stack(gate_out_list)
        # Length * Batch_size * hidden_dim
        return h_out, c_out, c_bar_out, decay_out, gate_out

    def conttime_loss(self, h, c, c_bar, decay, o, event_seqs, sim_duration, total_time_list, seq_len_lists,
                      time_simulation_index):
        batch_size = event_seqs.shape[0]
        # print(time_simulation_index)
        sim_len = time_simulation_index.shape[1]
        part_one_likelihood = torch.zeros(batch_size)
        sum_likelihood = torch.zeros(batch_size)

        # Get lambda
        type_intensity = torch.nn.functional.softplus(self.hidden_lambda(h)).transpose(0,1)
        for idx in range(batch_size):
            event_seq = event_seqs[idx]
            seq_len = seq_len_lists[idx]
            part_one_likelihood[idx] = torch.sum(torch.log(type_intensity[idx, torch.arange(seq_len), event_seq[1:seq_len+1]]))
            sum_likelihood[idx] = torch.sum(torch.log(torch.sum(type_intensity[idx, torch.arange(seq_len), :], dim=-1)))

        c_sim = []
        c_bar_sim = []
        decay_sim = []
        o_sim = []
        
        for j in range(batch_size):
            layer_c = c[time_simulation_index[j], j, :]
            c_sim.append(layer_c)
            layer_c_bar = c_bar[time_simulation_index[j], j, :]
            c_bar_sim.append(layer_c_bar)
            layer_decay = decay[time_simulation_index[j], j, :]
            decay_sim.append(layer_decay)
            layer_o = o[time_simulation_index[j], j, :]
            o_sim.append(layer_o)
        c_sim = torch.stack(c_sim).transpose(0,1)
        c_bar_sim = torch.stack(c_bar_sim).transpose(0,1)
        decay_sim = torch.stack(decay_sim).transpose(0,1)
        o_sim = torch.stack(o_sim).transpose(0,1)
        h_sim_list = []
        # print(c_sim)
        for idx in range(sim_duration.shape[1]):
            cell_next, h_sim = self.lstm_cell.decay(c_sim[idx], c_bar_sim[idx], decay_sim[idx], o_sim[idx], sim_duration[:, idx])
            h_sim_list.append(h_sim)
        h_sim_list = torch.stack(h_sim_list)
        sim_intensity = torch.nn.functional.softplus(self.hidden_lambda(h_sim_list)).transpose(0,1)
        part_two_likelihood = torch.zeros(batch_size)
        for idx in range(batch_size):
            coefficient = total_time_list[idx] / sim_len
            part_two_likelihood[idx] = torch.sum(torch.sum(sim_intensity[idx, torch.arange(sim_len), :])) * coefficient


        return part_one_likelihood, part_two_likelihood, sum_likelihood


In [ ]:
"""Validation, preprocessing, and helper functions for dataset conversion."""
import argparse
from collections import defaultdict
# import utils
import sys
import os
import datetime
import time
# from torch.utils.data import DataLoader
# import conttimefeat as conttime
import torch
import matplotlib.pyplot as plt
import test
import torch.nn.functional as F
import pandas as pd
import numpy as np

def log_valid(time_duration, type_test, seq_lens, test_features, device):
    """Compute log-likelihood based validation for a saved model.

    Args:
        time_duration (torch.Tensor): durations for validation.
        type_test (torch.Tensor): event types for validation.
        seq_lens (list[int]): sequence lengths.
        test_features (torch.Tensor): optional features.
        device (torch.device): device to run model on.

    Returns:
        tuple: (log_likelihood, type_likelihood, time_likelihood).
    """
    model = torch.load("model.pt", weights_only=False)
    seq_lens = torch.tensor(seq_lens)
    sim_durations, total_time_seqs, time_simulation_index = generate_simulation(time_duration, seq_lens)
    type_test.to(device)
    time_duration.to(device)
    sim_durations.to(device)
    total_time_seqs.to(device)
    seq_lens.to(device)
    time_simulation_index.to(device)
    h_out, c_out, c_bar_out, decay_out, gate_out = model(type_test, time_duration, test_features)
    part_one_likelihood, part_two_likelihood, sum_likelihood = model.conttime_loss(h_out, c_out, c_bar_out, decay_out, gate_out, type_test, sim_durations, total_time_seqs, seq_lens, time_simulation_index)
    total_size = torch.sum(seq_lens)
    log_likelihood = torch.sum(part_one_likelihood - part_two_likelihood) / total_size
    type_likelihood = torch.sum(part_one_likelihood - sum_likelihood) / total_size
    time_likelihood = log_likelihood - type_likelihood
    return log_likelihood, type_likelihood, time_likelihood

def type_valid(time_durations, seq_lens_lists, type_tests, test_features):
    """Predict next event type accuracy over test set using a saved model.

    Args:
        time_durations (torch.Tensor): durations per test item.
        seq_lens_lists (list[int]): sequence lengths.
        type_tests (torch.Tensor): type sequences.
        test_features (torch.Tensor): optional features.

    Returns:
        float: fraction of correct next-type predictions.
    """
    model = torch.load("model.pt", weights_only=False)
    numb_tests = time_durations.shape[0]
    original_types = []
    predicted_types = []
    for i in range(numb_tests):
        time_duration = time_durations[i:i+1]
        type_test = type_tests[i:i+1]
        seq_len = seq_lens_lists[i]
        time_feats = test_features[i:i+1]

        original_types.append(type_test[0][seq_len].item())
        type_test = type_test[:, :seq_len]
        time_duration = time_duration[:, :seq_len+1]

        h_out, c_out, c_bar_out, decay_out, gate_out = model(type_test, time_duration, time_feats)
        lambda_all = F.softplus(model.hidden_lambda(h_out[-1]))
        lambda_sum = torch.sum(lambda_all, dim=-1)
        lambda_all = lambda_all / lambda_sum
        _, predict_type = torch.max(lambda_all, dim=-1)
        predicted_types.append(predict_type.item())
    
    total_numb = len(original_types)
    numb_correct = 0
    for idx in range(total_numb):
        if predicted_types[idx] == original_types[idx]:
            numb_correct += 1
    return numb_correct / total_numb

def time_based_split(times, split_ratio=0.9):
    """Split sequences by a global time quantile cutoff.

    Args:
        times (list[tensor]): list of timestamp tensors.
        split_ratio (float): quantile cutoff for train/test.

    Returns:
        tuple: (train_times, test_times) lists with same structure as input.
    """
    all_times = torch.cat(times)
    cutoff = torch.quantile(all_times, split_ratio)
    
    train_times = []
    test_times = []
    
    for seq in times:
        mask_train = seq <= cutoff
        mask_test = seq > cutoff
        train_times.append(seq[mask_train])
        test_times.append(seq[mask_test])
    
    return train_times, test_times

def preprocess_midprice_to_neuralhawkes_feat(csv_path, percent=0.0001):
    """Preprocess LOB midprice CSV into Neural Hawkes events and per-event features.

    Args:
        csv_path (str): path to CSV containing 'time' and 'midprice' columns and feature columns.
        percent (float): fractional threshold for detecting a price change event.

    Returns:
        tuple: (event_times, event_types, event_features) as tensors.
    """
    df = pd.read_csv(csv_path)
    df['timestamp'] = pd.to_datetime(df['time'])
    df = df.sort_values('timestamp').reset_index(drop=True)
    
    feature_cols = [f'bid_price_{i}' for i in range(1,11)]+[f'ask_price_{i}' for i in range(1,11)]+[f'bid_volume_{i}' for i in range(1,11)]+[f'ask_volume_{i}' for i in range(1,11)]
    
    event_times = []
    event_types = []
    event_features = []
    
    prev_price = df.loc[0, 'midprice']
    t0 = df.loc[0, 'timestamp']
    
    for i in range(1, len(df)):
        price = df.loc[i, 'midprice']
        time = df.loc[i, 'timestamp']
        
        if price != prev_price:  # event occurs when price changes
            dt = (time - t0).total_seconds()
            if(price > prev_price*(1+percent)):
                event_times.append(dt)
                event_types.append(0)
                event_features.append(df.loc[i-1, feature_cols].values.astype(float))
            elif(price < prev_price*(1-percent)):
                event_times.append(dt)
                event_types.append(1)
                event_features.append(df.loc[i-1, feature_cols].values.astype(float))
            prev_price = price
    
    if len(event_times) < 2:
        raise ValueError("Not enough events detected in data.")
        
    return torch.tensor(event_times), torch.tensor(event_types), torch.tensor(event_features)

def combine_by_type_feat(event_times, event_types, feats):
    """Group event times and features by event type.

    Args:
        event_times (iterable): list/array of event timestamps.
        event_types (iterable): list/array of integer types.
        feats (iterable): list/array of feature vectors.

    Returns:
        tuple: (comb_eventtimes, comb_feats) lists of tensors grouped by type.
    """
    grouped_times = defaultdict(list)
    grouped_feats = defaultdict(list)

    for t, typ, f in zip(event_times, event_types, feats):
        grouped_times[typ].append(t)
        grouped_feats[typ].append(f)

    types = sorted(grouped_times.keys())
    
    comb_eventtimes = [torch.tensor(grouped_times[typ], dtype=torch.float32) for typ in types]
    comb_feats = [torch.tensor(grouped_feats[typ], dtype=torch.float32) for typ in types]
    
    return comb_eventtimes, comb_feats

def standard_normalize_feat(tensor_list):
    """Standardize a list/array of feature vectors (zero mean, unit std).

    Args:
        tensor_list (iterable): iterable of numeric feature vectors.

    Returns:
        torch.Tensor: normalized tensor of features.
    """
    X = torch.tensor(np.array(tensor_list))

    X = torch.nan_to_num(X, nan=0.0)

    mean = X.mean(dim=0, keepdim=True)
    std = X.std(dim=0, keepdim=True)
    std[std == 0] = 1

    X_norm = (X - mean) / std

    return torch.tensor(X_norm)

def time_based_split_feat(times, feats, split_ratio=0.9):
    """Split times and corresponding features by a global time cutoff.

    Returns train_times, test_times, train_feats, test_feats.
    """
    all_times = torch.cat(times)
    cutoff = torch.quantile(all_times, split_ratio)
    
    train_times = []
    test_times = []
    train_feats = []
    test_feats = []
    
    for i in range(len(times)):
        seq = times[i]
        mask_train = seq <= cutoff
        mask_test = seq > cutoff
        train_times.append(seq[mask_train])
        test_times.append(seq[mask_test])

        feat = feats[i]
        train_feats.append(feat[mask_train])
        test_feats.append(feat[mask_test])
    
    return train_times, test_times, train_feats, test_feats

In [ ]:
"""Evaluation helpers that compute RMSE using Monte Carlo estimation."""
def normal_rmse(time_duration, seq_lens_list, type_test, n_samples, model):
    """Estimate RMSE for next-event timing prediction using Monte Carlo simulation.

    Args:
        time_duration (torch.Tensor): Sequence of observed inter-arrival times.
        seq_lens_list (list[int]): Sequence lengths.
        type_test (torch.Tensor): Event type tensor.
        n_samples (int): Number of Monte Carlo samples.
        model (nn.Module): Trained event-time model.

    Returns:
        float: Root mean squared error of predicted event times.
    """
    max_len = time_duration.shape[-1]
    estimated_times = []
    original_time = time_duration[0][57:].tolist()

    for idx in range(57, max_len):
        time_durations = time_duration[0][:idx]
        type_tests = type_test[0][:idx]
        max_duration = torch.max(time_durations)
        simulated_duration = torch.sort(torch.empty(n_samples).uniform_(0, 40 * max_duration.item()))[0].reshape(n_samples, 1)
        time_durations = time_durations.expand(n_samples, time_durations.shape[-1])
        time_duration_sim_padded = torch.cat((time_durations, simulated_duration), dim=1)
        type_tests = type_tests.expand(n_samples, type_tests.shape[-1])

        h_out, c_out, c_bar_out, decay_out, gate_out = model(type_tests, time_duration_sim_padded)
        simulated_h, simulated_c, simulated_c_bar, simulated_decay, simulated_o = h_out[-1], c_out[-1], c_bar_out[-1], decay_out[-1], gate_out[-1]
        h_last, c_last, c_bar_last, decay_last, o_last = h_out[-2][0], c_out[-2][0], c_bar_out[-2][0], decay_out[-2][0], gate_out[-2][0]

        estimated_lambda_sum = torch.sum(F.softplus(model.hidden_lambda(simulated_h)), dim=-1)
        estimated_lambda_sum = estimated_lambda_sum.reshape(n_samples)
        simulated_duration = simulated_duration.reshape(n_samples)
        simulated_integral_exp_terms = torch.stack([(torch.sum(estimated_lambda_sum[:(i+1)]) * (simulated_duration[i] / (i+1))) for i in range(0, n_samples)])
        simulated_density = estimated_lambda_sum * torch.exp(-simulated_integral_exp_terms)
        estimated_time = torch.sum(simulated_duration * simulated_density) * (40 * max_duration.item()) / n_samples
        estimated_times.append(estimated_time.item())

    rmse = sqrt(mean_squared_error(original_time, estimated_times))
    return rmse


def feat_rmse(time_duration, seq_lens_list, feat_list, type_test, n_samples, model):
    """Estimate RMSE for a feature-aware CTLSTM using Monte Carlo timing simulation.

    Args:
        time_duration (torch.Tensor): Inter-arrival time sequence.
        seq_lens_list (list[int]): Sequence lengths.
        feat_list (list): List of feature vectors corresponding to the sequence.
        type_test (torch.Tensor): Event type sequence.
        n_samples (int): Number of Monte Carlo samples.
        model (nn.Module): Trained feature-aware model.

    Returns:
        float: RMSE between observed and predicted event times.
    """
    max_len = time_duration.shape[-1]
    estimated_times = []
    estimated_intensities = []
    estimated_types = []
    original_time = time_duration[0][75:].tolist()

    for idx in range(75, max_len):
        time_durations = time_duration[0][idx - 75:idx]
        type_tests = type_test[0][idx - 75:idx]
        feats = feat_list[0][idx - 75:idx]
        feats = torch.tensor(np.stack(feats), dtype=torch.float32)

        max_duration = torch.max(time_durations)
        simulated_duration = torch.sort(torch.empty(n_samples).uniform_(0, 40 * max_duration.item()))[0].reshape(n_samples, 1)
        time_durations = time_durations.expand(n_samples, time_durations.shape[-1])
        time_duration_sim_padded = torch.cat((time_durations, simulated_duration), dim=1)
        type_tests = type_tests.expand(n_samples, type_tests.shape[-1])

        T, n_feat = feats.shape
        feats_expanded = feats.unsqueeze(0).expand(n_samples, T, n_feat)
        sim_feat = torch.zeros(n_samples, 1, n_feat)
        feats_sim_padded = torch.cat((feats_expanded, sim_feat), dim=1)

        h_out, c_out, c_bar_out, decay_out, gate_out = model(type_tests, time_duration_sim_padded, feats_sim_padded)
        simulated_h, simulated_c, simulated_c_bar, simulated_decay, simulated_o = h_out[-1], c_out[-1], c_bar_out[-1], decay_out[-1], gate_out[-1]
        h_last, c_last, c_bar_last, decay_last, o_last = h_out[-2][0], c_out[-2][0], c_bar_out[-2][0], decay_out[-2][0], gate_out[-2][0]

        # calculate estimated lambda and density
        # Use of monte carlo simulation of the way below:
        # p(di) = lambda(di) * exp(-(sum from 1 to i (lambda(dk))) * di / i) to calculate p(di) = lambda(di) * exp(-(integral from 0 to di (lambda(tao))))
        estimated_lambda_sum = torch.sum(F.softplus(model.hidden_lambda(simulated_h)), dim=-1)
        estimated_lambda_sum = estimated_lambda_sum.reshape(n_samples)
        simulated_duration = simulated_duration.reshape(n_samples)
        simulated_integral_exp_terms = torch.stack([(torch.sum(estimated_lambda_sum[:(i+1)]) * (simulated_duration[i] / (i+1))) for i in range(0, n_samples)])
        simulated_density = estimated_lambda_sum * torch.exp(-simulated_integral_exp_terms)
        estimated_time = torch.sum(simulated_duration * simulated_density) * (40 * max_duration.item()) / n_samples
        estimated_times.append(estimated_time.item())

    rmse = sqrt(mean_squared_error(original_time, estimated_times))
    return rmse

In [ ]:
"""Experiment hyperparameters and dataset root path."""
lr = 0.01
seq_len = 75
num_epochs = 20
batch_size = 32
used_model = False
type_size = 2
split_ratio = 0.75
num_files = 60
percent=0.0001
root_dir = '/kaggle/input/lob-1sec-dataset/PZU/'

# Original

In [ ]:
def log_valid_old(time_duration, type_test, seq_lens, device):
    model = torch.load("model_old.pt", weights_only=False)
    seq_lens = torch.tensor(seq_lens)
    sim_durations, total_time_seqs, time_simulation_index = generate_simulation(time_duration, seq_lens)
    type_test.to(device)
    time_duration.to(device)
    sim_durations.to(device)
    total_time_seqs.to(device)
    seq_lens.to(device)
    time_simulation_index.to(device)
    h_out, c_out, c_bar_out, decay_out, gate_out = model(type_test, time_duration)
    part_one_likelihood, part_two_likelihood, sum_likelihood = model.conttime_loss(h_out, c_out, c_bar_out, decay_out, gate_out, type_test, sim_durations, total_time_seqs, seq_lens, time_simulation_index)
    total_size = torch.sum(seq_lens)
    log_likelihood = torch.sum(part_one_likelihood - part_two_likelihood) / total_size
    type_likelihood = torch.sum(part_one_likelihood - sum_likelihood) / total_size
    time_likelihood = log_likelihood - type_likelihood
    return log_likelihood, type_likelihood, time_likelihood

def type_valid_old(time_durations, seq_lens_lists, type_tests):
    model = torch.load("model_old.pt", weights_only=False)
    numb_tests = time_durations.shape[0]
    original_types = []
    predicted_types = []
    for i in range(numb_tests):
        time_duration = time_durations[i:i+1]
        type_test = type_tests[i:i+1]
        seq_len = seq_lens_lists[i]

        original_types.append(type_test[0][seq_len].item())
        type_test = type_test[:, :seq_len]
        time_duration = time_duration[:, :seq_len+1]

        h_out, c_out, c_bar_out, decay_out, gate_out = model(type_test, time_duration)
        lambda_all = F.softplus(model.hidden_lambda(h_out[-1]))
        lambda_sum = torch.sum(lambda_all, dim=-1)
        lambda_all = lambda_all / lambda_sum
        # print(lambda_all)
        _, predict_type = torch.max(lambda_all, dim=-1)
        predicted_types.append(predict_type.item())
    
    total_numb = len(original_types)
    numb_correct = 0
    for idx in range(total_numb):
        if predicted_types[idx] == original_types[idx]:
            numb_correct += 1
    return numb_correct / total_numb

def time_based_split(times, split_ratio=0.9):
    # Flatten and find global cutoff
    all_times = torch.cat(times)
    cutoff = torch.quantile(all_times, split_ratio)
    
    train_times = []
    test_times = []
    
    for seq in times:
        mask_train = seq <= cutoff
        mask_test = seq > cutoff
        train_times.append(seq[mask_train])
        test_times.append(seq[mask_test])
    
    return train_times, test_times

def combine_by_type(event_times, event_types, feats):
    """
    event_times: list[float]
    event_types: list[int]
    feats: list[list or tensor]
    
    Returns:
        comb_eventtimes: list[torch.Tensor]  # grouped by type
        comb_feats: list[torch.Tensor]       # grouped by type
    """
    grouped_times = defaultdict(list)
    grouped_feats = defaultdict(list)

    for t, typ, f in zip(event_times, event_types, feats):
        grouped_times[typ].append(t)
        grouped_feats[typ].append(f)

    # ensure consistent ordering by event type
    types = sorted(grouped_times.keys())
    
    comb_eventtimes = [torch.tensor(grouped_times[typ], dtype=torch.float32) for typ in types]
    comb_feats = [torch.tensor(grouped_feats[typ], dtype=torch.float32) for typ in types]
    
    return comb_eventtimes, comb_feats

def standard_normalize_list(tensor_list):
    # Stack into a single tensor: shape (N, 20)
    X = torch.tensor(np.array(tensor_list))

    # Replace NaNs with 0
    X = torch.nan_to_num(X, nan=0.0)

    # Compute mean & std per feature (dim=0)
    mean = X.mean(dim=0, keepdim=True)
    std = X.std(dim=0, keepdim=True)
    # Avoid division by zero
    std[std == 0] = 1

    # Standard normalization: (x - mean)/std
    X_norm = (X - mean) / std

    # Return as list of tensors again
    return [row.numpy() for row in X_norm]

# if __name__ == "__main__":
    # parser = argparse.ArgumentParser(description="Training model..")
    # parser.add_argument("--lr", type=float, default=0.01, help="learning rate")
    # parser.add_argument("--epochs", type=int, default=10, help="maximum epochs")
    # parser.add_argument("--seq_len", type=int, default=-1, help="truncated sequence length for hawkes and self-correcting, -1 means full sequence")
    # parser.add_argument("--batch_size", type=int, default=10, help="Batch_size for each train iteration")
    # parser.add_argument("--used_past_model", type=bool, help="True to use a trained model named model_old.pt")

    # config = parser.parse_args()

    # lr = config.lr
    # seq_len = config.seq_len
    # num_epochs = config.epochs
    # batch_size = config.batch_size
    # used_model = config.used_past_model
train_lle=[]
test_lle=[]
test_acc=[]
rmse_train_full=[]
rmse_test_full=[]
for num_run in range(5):
    now = str(datetime.datetime.today()).split()
    now = now[0]+"-"+now[1][:5]
    id_process = os.getpid()
    print("id: " + str(id_process))
    os.makedirs('./logs', exist_ok=True)
    log_file_name = "./logs/train_process"+str(id_process)+".txt"
    log = open(log_file_name, 'w')
    log.write("Data when training: " + str(datetime.datetime.now()))
    log.write("\nTraining-id: " + str(id_process))
    log.write("\nLearning rate: " + str(lr))
    log.write("\nMax epochs: " + str(num_epochs))
    log.write("\nseq lens: " + str(seq_len))
    log.write("\nbatch size for train: " + str(batch_size))
    log.write("\nuse previous model: " + str(used_model))

    t1 = time.time()
    print("Processing data...")

    files=os.listdir(root_dir)[:num_files]
    temp_time_duration, event_types, event_features = preprocess_midprice_to_neuralhawkes_feat(root_dir+files[0], percent=percent)
    time_duration = torch.cat([
        temp_time_duration[:1],
        torch.diff(temp_time_duration)
    ])
    for fn in files[1:]:
        temp_time_duration, temp_event_types, temp_event_features = preprocess_midprice_to_neuralhawkes_feat(root_dir+fn, percent=percent)
        inter_arrival = torch.cat([
            temp_time_duration[:1],
            torch.diff(temp_time_duration)
        ])
        time_duration = torch.cat([time_duration, inter_arrival])
        event_types = torch.cat([event_types, temp_event_types])
        event_features = torch.cat([event_features, temp_event_features])
    # time_duration = [torch.tensor(times),]
    event_features = standard_normalize_list(event_features)
    # 1. Read times and sequence lengths
    # time_duration, seq_lens_list = utils.open_txt_file("output_neuralhawkes/time.txt")
    # time_duration, event_types, event_features = preprocess_midprice_to_neuralhawkes_feat('datamin.csv')
    # time_duration = [torch.tensor(times),]
    # event_features = standard_normalize_list(event_features)
    # print([len(d) for d in time_duration], seq_lens_list)
    # 2. Create type list as unstacked list of tensors (same length as time_duration)
    # type_train = [torch.zeros_like(d, dtype=torch.long) for d in time_duration]

    # type_size = 2

    # 3. Pad sequences
    # time_train, time_test = time_based_split(time_duration, split_ratio=0.9)

    split_len = int(0.7*len(time_duration))
    time_train, time_test = [time_duration[:split_len],], [time_duration[split_len:],]
    type_train_seq, type_test_seq = [event_types[:split_len],], [event_types[split_len:],]
    feats_train, feats_test = [event_features[:split_len],], [event_features[split_len:],]

    seq_lens_train = [len(i) for i in time_train]
    seq_lens_test = [len(i) for i in time_test]

    print("Train seqs after split: ", seq_lens_train)
    print("Test seqs after split: ", seq_lens_test)

    # type_train_seq = [torch.zeros_like(time_train[i], dtype=torch.long)+i for i in range(len(time_train))]
    # type_test_seq = [torch.zeros_like(time_test[i]+i, dtype=torch.long) for i in range(len(time_test))]
    # type_train_seq = utils.get_index_txt(time_train)
    # type_test_seq = utils.get_index_txt(time_test)
    # if seq_len == -1:
    #     time_duration, type_train = utils.padding_full(time_duration, type_train, seq_lens_list, type_size)
    # else:
    #     time_duration, type_train, seq_lens_list = utils.padding_seq_len(time_duration, type_train, type_size, seq_len)
    

    # --------------------------
    # Padding
    # --------------------------
    if seq_len == -1:
        train_duration, type_train = padding_full(time_train, type_train_seq, seq_lens_train, type_size)
        test_duration, type_test = padding_full(time_test, type_test_seq, seq_lens_test, type_size)
    else:
        train_duration, type_train, seq_lens_train = padding_seq_len(time_train, type_train_seq, type_size, seq_len)
        test_duration, type_test, seq_lens_test = padding_seq_len(time_test, type_test_seq, type_size, seq_len)

    # print("Train events:")
    # print(time_duration.shape)

    print(train_duration.shape, test_duration.shape)
    # If seqlen==-1: [2, 812], [2, 89]
    # If seqlen==10: train [1512, 11], [152, 11]
    # exit(0)

    # print("Before batch: ", time_duration, type_train, seq_lens_train)
    train_data = Data_Batch(train_duration, type_train, seq_lens_train)
    # print([(i, train_data[0][i]) for i in train_data[1]])
    # print(train_data[1]['event_seq'])
    train_data = DataLoader(train_data, batch_size=batch_size, shuffle=True)

    print("Data Processing Finished...")
    t2 = time.time()
    data_process_time = t2 - t1
    print("Getting data takes: " + str(data_process_time) + " seconds")
    log.write("\n\nGetting data takes: " + str(data_process_time) + " seconds")

    print("start training...")
    t3 = time.time()
    if used_model:
        model = torch.load("model_old.pt")
    else:
        model = Conttime(n_types=type_size, lr=lr)

    if torch.cuda.is_available():
        device = torch.device('cuda')
        print("You are using GPU acceleration.")
        log.write("\nYou are using GPU acceleration.")
        print("Number of GPU: ", torch.get_num_threads())
        log.write("\n\nNumber of GPU: " + str((torch.get_num_threads())))
    else:
        device = torch.device("cpu")
        print("CUDA is not Available. You are using CPU only.")
        log.write("\nCUDA is not Available. You are using CPU only.")
        print("Number of cores: ", os.cpu_count())
        log.write("\n\nNumber of cores: " + str(os.cpu_count()))

    loss_value = []

    log_test_list = []
    log_time_list = []
    log_type_list = []
    rmse_train_list = []
    rmse_test_list = []
    type_accuracy_list = []
    for i in range(num_epochs):
        loss_total = 0
        events_total = 0
        max_len = len(train_data)
        for idx, a_batch in enumerate(train_data):
            durations, type_items, seq_lens = a_batch['duration_seq'], a_batch['event_seq'], a_batch['seq_len']
            # print("Batch shapes: ", durations.shape, type_items.shape, seq_lens.shape)
            # print("Durations:", durations.shape)
            # print("Type items:", type_items)

            sim_durations, total_time_seqs, time_simulation_index = generate_simulation(durations, seq_lens)

            type_items.to(device)
            durations.to(device)
            sim_durations.to(device)
            total_time_seqs.to(device)
            seq_lens.to(device)
            time_simulation_index.to(device)
            batch = (type_items, durations)
            loss = model.train_batch(batch, sim_durations, total_time_seqs, seq_lens, time_simulation_index)
            log_likelihood = -loss
            total_size = torch.sum(seq_lens)
            loss_total += log_likelihood.item()
            events_total += total_size.item()
            # print("In epochs {0}, process {1} over {2} is done".format(i, idx, max_len))
        avg_log = loss_total / events_total
        loss_value.append(-avg_log)
        print("The log-likelihood at epochs {0} is {1}".format(i, avg_log))
        log.write("\nThe log likelihood at epochs {0} is {1}".format(i, avg_log))
        print("model saved..")
        torch.save(model, "model_old.pt")

        print("\nvalidating on log likelihood...")
        log_likelihood, type_likelihood, time_likelihood = log_valid_old(test_duration, type_test, seq_lens_test, device)
        log_test_list.append(-log_likelihood.item())
        log_type_list.append(-type_likelihood.item())
        log_time_list.append(-time_likelihood.item())
        print("Test log likelihood:", log_likelihood)
        print("\nvalidating on type prediction accuracy if we know when will next event happens...\n\n")
        accuracy = type_valid_old(test_duration, seq_lens_test, type_test)
        print("Accuracy: ", accuracy)
        
        type_accuracy_list.append(accuracy)
        
        train_rmse = normal_rmse(train_duration, seq_lens_train, type_train, 10000, model)
        test_rmse = normal_rmse(test_duration, seq_lens_test, type_test, 10000, model)
        print(f"Train RMSE: {train_rmse}")
        print(f"Test RMSE: {test_rmse}")
        rmse_train_list.append(train_rmse)
        rmse_test_list.append(test_rmse)

    train_lle.append(loss_value)
    test_lle.append(log_test_list)
    test_acc.append(type_accuracy_list)
    rmse_train_full.append(rmse_train_list)
    rmse_test_full.append(rmse_test_list)

    figure, ax = plt.subplots(nrows=1, ncols=3, figsize=(16, 4))
    ax[0].set_xlabel("epochs")
    ax[0].plot(loss_value, label='training loss')
    ax[0].plot(log_test_list, label='testing loss')
    ax[0].legend()
    ax[1].set_xlabel("epochs")
    ax[1].plot(log_type_list, label='testing type loss')
    ax[1].plot(log_time_list, label='testing time loss')
    ax[1].legend()
    ax[2].set_xlabel("epochs")
    ax[2].set_ylabel('accuracy')
    ax[2].set_title('type-validation-accuracy')
    ax[2].plot(type_accuracy_list, label='dev type accuracy')
    plt.subplots_adjust(top=0.85)
    figure.tight_layout()
    plt.savefig("training_simple.jpg")

    t4 = time.time()
    training_time = t4 - t3
    print("training done..")
    print("training takes {0} seconds".format(training_time))
    log.write("\ntraining takes {0} seconds".format(training_time))
    log.close()

    print("Saving training loss and validation data...")
    print("If you have a trained model before this, please combine the previous train_date file to" +
        " generate plots that are able to show the whole training information")
    training_info_file = "training-data-" + now + ".txt"
    file = open(training_info_file, 'w')
    file.write("log-likelihood: ")
    file.writelines(str(item) + " " for item in loss_value)
    file.write('\nlog-test-likelihood: ')
    file.writelines(str(item) + " " for item in log_test_list)
    file.write('\nlog-type-likelihood: ')
    file.writelines(str(item) + " " for item in log_type_list)
    file.write('\nlog-time-likelihood: ')
    file.writelines(str(item) + " " for item in log_time_list)
    file.write('\naccuracy: ')
    file.writelines(str(item) + " " for item in type_accuracy_list)
    file.close()
    print("Every works are done!")

# FeatLSTM

In [ ]:
train_lle=[]
test_lle=[]
test_acc=[]
rmse_train_full=[]
rmse_test_full=[]
for num_run in range(5):
    now = str(datetime.datetime.today()).split()
    now = now[0]+"-"+now[1][:5]
    id_process = os.getpid()
    print("id: " + str(id_process))
    os.makedirs('./logs', exist_ok=True)
    log_file_name = "./logs/train_process"+str(id_process)+".txt"
    log = open(log_file_name, 'w')
    log.write("Data when training: " + str(datetime.datetime.now()))
    log.write("\nTraining-id: " + str(id_process))
    log.write("\nLearning rate: " + str(lr))
    log.write("\nMax epochs: " + str(num_epochs))
    log.write("\nseq lens: " + str(seq_len))
    log.write("\nbatch size for train: " + str(batch_size))
    log.write("\nuse previous model: " + str(used_model))
    
    t1 = time.time()
    print("Processing data...")
    
    # 1. Read times and sequence lengths
    files=os.listdir(root_dir)[:num_files]
    temp_time_duration, event_types, event_features = preprocess_midprice_to_neuralhawkes_feat(root_dir+files[0], percent=percent)
    time_duration = torch.cat([
        temp_time_duration[:1],
        torch.diff(temp_time_duration)
    ])
    for fn in files[1:]:
        temp_time_duration, temp_event_types, temp_event_features = preprocess_midprice_to_neuralhawkes_feat(root_dir+fn, percent=percent)
        inter_arrival = torch.cat([
            temp_time_duration[:1],
            torch.diff(temp_time_duration)
        ])
        time_duration = torch.cat([time_duration, inter_arrival])
        event_types = torch.cat([event_types, temp_event_types])
        event_features = torch.cat([event_features, temp_event_features])
    # time_duration = [torch.tensor(times),]
    event_features = standard_normalize_feat(event_features)
    
    # # print([len(d) for d in time_duration], seq_lens_list)
    # # 2. Create type list as unstacked list of tensors (same length as time_duration)
    # type_train = [torch.zeros_like(d, dtype=torch.long) for d in time_duration]
    
    # # 3. Pad sequences
    # time_train, time_test = time_based_split(time_duration, split_ratio=0.9)
    print("Input shapes:", time_duration.shape, event_types.shape, event_features.shape)
    split_len = int(split_ratio*len(time_duration))
    time_train, time_test = [time_duration[:split_len],], [time_duration[split_len:],]
    type_train_seq, type_test_seq = [event_types[:split_len],], [event_types[split_len:],]
    train_features, test_features = [event_features[:split_len],], [event_features[split_len:],]
    
    # exit(0)
    # print([len(d) for d in time_duration], seq_lens_list)
    # 2. Create type list as unstacked list of tensors (same length as time_duration)
    # type_train = [torch.zeros_like(d, dtype=torch.long) for d in train_duration]
    # type_test = [torch.zeros_like(d, dtype=torch.long) for d in test_duration]
    
    # 3. Pad sequences
    seq_lens_train = [len(i) for i in time_train]
    seq_lens_test = [len(i) for i in time_test]
    
    print("Train after split: ", seq_lens_train)
    print("Test after split: ", seq_lens_test)
    
    # if seq_len == -1:
    #     time_duration, type_train = utils.padding_full(time_duration, type_train, seq_lens_list, type_size)
    # else:
    #     time_duration, type_train, seq_lens_list = utils.padding_seq_len(time_duration, type_train, type_size, seq_len)
    
    
    # --------------------------
    # Padding
    # --------------------------
    if seq_len == -1:
        # time_train, type_train_seq, seq_lens_train, type_size
        # time_duration, type_train, event_features = utils.padding_full_feats(time_duration, type_train, seq_lens_train, type_size, event_features)
        train_duration, type_train, train_features = padding_full_feats(time_train, type_train_seq, seq_lens_train, type_size, train_features)
        test_duration, type_test, test_features = padding_full_feats(time_test, type_test_seq, seq_lens_test, type_size, test_features)
    else:
        train_duration, type_train, seq_lens_train, train_features = padding_seq_len_feats(time_train, type_train_seq, type_size, seq_len, train_features)
        test_duration, type_test, seq_lens_test, test_features = padding_seq_len_feats(time_test, type_test_seq, type_size, seq_len, test_features)
    
    # Output would have been:
    # print("Before batch: ", train_duration.shape, type_train.shape, seq_lens_train, train_features.shape)
    
    train_data = Data_Batch_Feat(train_duration, type_train, seq_lens_train, train_features)
    train_data = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    
    print("Data Processing Finished...")
    t2 = time.time()
    data_process_time = t2 - t1
    print("Getting data takes: " + str(data_process_time) + " seconds")
    log.write("\n\nGetting data takes: " + str(data_process_time) + " seconds")
    
    print("start training...")
    t3 = time.time()
    if used_model:
        model = torch.load("model.pt")
    else:
        model = ConttimeFeat(n_types=type_size, lr=lr, extra_dim=40)
    
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print("You are using GPU acceleration.")
        log.write("\nYou are using GPU acceleration.")
        print("Number of GPU: ", torch.get_num_threads())
        log.write("\n\nNumber of GPU: " + str((torch.get_num_threads())))
    else:
        device = torch.device("cpu")
        print("CUDA is not Available. You are using CPU only.")
        log.write("\nCUDA is not Available. You are using CPU only.")
        print("Number of cores: ", os.cpu_count())
        log.write("\n\nNumber of cores: " + str(os.cpu_count()))
    
    loss_value = []
    
    log_test_list = []
    log_time_list = []
    log_type_list = []
    rmse_train_list = []
    rmse_test_list = []
    type_accuracy_list = []
    for i in range(num_epochs):
        loss_total = 0
        events_total = 0
        max_len = len(train_data)
        for idx, a_batch in enumerate(train_data):
            durations, type_items, seq_lens, feats = a_batch['duration_seq'], a_batch['event_seq'], a_batch['seq_len'], a_batch['features']
            # The shapes when seqlen=-1: durations:(2, 901), typeitems:(2, 901), seqlens: (2)
            # The shapes when seqlen=10: 
            # print("Batch details:", durations.shape, type_items.shape, seq_lens.shape, feats.shape)
            # print(durations)
            # print(type_items)
            # print(seq_lens)
            # exit(0)
    
            sim_durations, total_time_seqs, time_simulation_index = generate_simulation(durations, seq_lens)
    
            type_items.to(device)
            durations.to(device)
            sim_durations.to(device)
            total_time_seqs.to(device)
            seq_lens.to(device)
            time_simulation_index.to(device)
            feats.to(device)
    
            batch = (type_items, durations, feats)
            loss = model.train_batch(batch, sim_durations, total_time_seqs, seq_lens, time_simulation_index)
            log_likelihood = -loss
            total_size = torch.sum(seq_lens)
            loss_total += log_likelihood.item()
            events_total += total_size.item()
            # print("In epochs {0}, process {1} over {2} is done".format(i, idx, max_len))
        avg_log = loss_total / events_total
        loss_value.append(-avg_log)
        print("The log-likelihood at epochs {0} is {1}".format(i, avg_log))
        log.write("\nThe log likelihood at epochs {0} is {1}".format(i, avg_log))
        print("model saved..")
        torch.save(model, "model.pt")
    
        print("\nvalidating on log likelihood...")
        log_likelihood, type_likelihood, time_likelihood = log_valid(test_duration, type_test, seq_lens_test, test_features, device)
        print("Test log likelihood:", log_likelihood.item())
        log_test_list.append(-log_likelihood.item())
        log_type_list.append(-type_likelihood.item())
        log_time_list.append(-time_likelihood.item())
    
        print("\nvalidating on type prediction accuracy if we know when will next event happens...\n\n")
        accuracy = type_valid(test_duration, seq_lens_test, type_test, test_features)
        type_accuracy_list.append(accuracy)
        print("Accuracy:", accuracy)

        train_rmse = feat_rmse(train_duration, seq_lens_train, train_features, type_train, 10000, model)
        test_rmse = feat_rmse(test_duration, seq_lens_test, test_features, type_test, 10000, model)
        print(f"Train RMSE: {train_rmse}")
        print(f"Test RMSE: {test_rmse}")
        rmse_train_list.append(train_rmse)
        rmse_test_list.append(test_rmse)

    train_lle.append(loss_value)
    test_lle.append(log_test_list)
    test_acc.append(type_accuracy_list)
    rmse_train_full.append(rmse_train_list)
    rmse_test_full.append(rmse_test_list)
    
    figure, ax = plt.subplots(nrows=1, ncols=3, figsize=(16, 4))
    ax[0].set_xlabel("epochs")
    ax[0].plot(loss_value, label='training loss')
    ax[0].plot(log_test_list, label='testing loss')
    ax[0].legend()
    ax[1].set_xlabel("epochs")
    ax[1].plot(log_type_list, label='testing type loss')
    ax[1].plot(log_time_list, label='testing time loss')
    ax[1].legend()
    ax[2].set_xlabel("epochs")
    ax[2].set_ylabel('accuracy')
    ax[2].set_title('type-validation-accuracy')
    ax[2].plot(type_accuracy_list, label='dev type accuracy')
    plt.subplots_adjust(top=0.85)
    figure.tight_layout()
    plt.savefig("training.jpg")
    
    t4 = time.time()
    training_time = t4 - t3
    print("training done..")
    print("training takes {0} seconds".format(training_time))
    log.write("\ntraining takes {0} seconds".format(training_time))
    log.close()
    
    print("Saving training loss and validation data...")
    print("If you have a trained model before this, please combine the previous train_date file to" +
        " generate plots that are able to show the whole training information")
    training_info_file = "training-data-" + now + ".txt"
    file = open(training_info_file, 'w')
    file.write("log-likelihood: ")
    file.writelines(str(item) + " " for item in loss_value)
    file.write('\nlog-test-likelihood: ')
    file.writelines(str(item) + " " for item in log_test_list)
    file.write('\nlog-type-likelihood: ')
    file.writelines(str(item) + " " for item in log_type_list)
    file.write('\nlog-time-likelihood: ')
    file.writelines(str(item) + " " for item in log_time_list)
    file.write('\naccuracy: ')
    file.writelines(str(item) + " " for item in type_accuracy_list)
    file.close()
    print("Every works are done!")

# Attn-LSTM

In [ ]:
"""Attention LSTM full training loop and evaluation (executable)."""
train_lle=[]
test_lle=[]
test_acc=[]
rmse_train_full=[]
rmse_test_full=[]
for num_run in range(4):
    now = str(datetime.datetime.today()).split()
    now = now[0]+"-"+now[1][:5]
    id_process = os.getpid()
    print("id: " + str(id_process))
    print(f"Attempt: {num_run}")
    os.makedirs('./logs', exist_ok=True)
    log_file_name = "./logs/train_process"+str(id_process)+".txt"
    log = open(log_file_name, 'w')
    log.write("Data when training: " + str(datetime.datetime.now()))
    log.write("\nTraining-id: " + str(id_process))
    log.write("\nLearning rate: " + str(lr))
    log.write("\nMax epochs: " + str(num_epochs))
    log.write("\nseq lens: " + str(seq_len))
    log.write("\nbatch size for train: " + str(batch_size))
    log.write("\nuse previous model: " + str(used_model))
    
    t1 = time.time()
    print("Processing data...")
    
    # 1. Read times and sequence lengths
    files=os.listdir(root_dir)[:num_files]
    temp_time_duration, event_types, event_features = preprocess_midprice_to_neuralhawkes_feat(root_dir+files[0], percent=percent)
    time_duration = torch.cat([
        temp_time_duration[:1],
        torch.diff(temp_time_duration)
    ])
    for fn in files[1:]:
        temp_time_duration, temp_event_types, temp_event_features = preprocess_midprice_to_neuralhawkes_feat(root_dir+fn, percent=percent)
        inter_arrival = torch.cat([
            temp_time_duration[:1],
            torch.diff(temp_time_duration)
        ])
        time_duration = torch.cat([time_duration, inter_arrival])
        event_types = torch.cat([event_types, temp_event_types])
        event_features = torch.cat([event_features, temp_event_features])
    # time_duration = [torch.tensor(times),]
    event_features = standard_normalize_feat(event_features)
    
    # print([len(d) for d in time_duration], seq_lens_list)
    # 2. Create type list as unstacked list of tensors (same length as time_duration)
    # type_train = [torch.zeros_like(d, dtype=torch.long) for d in time_duration]
    
    # 3. Pad sequences
    # time_train, time_test = time_based_split(time_duration, split_ratio=0.9)
    print("Input shapes:", time_duration.shape, event_types.shape, event_features.shape)
    split_len = int(split_ratio*len(time_duration))
    time_train, time_test = [time_duration[:split_len],], [time_duration[split_len:],]
    type_train_seq, type_test_seq = [event_types[:split_len],], [event_types[split_len:],]
    train_features, test_features = [event_features[:split_len],], [event_features[split_len:],]
    
    # # exit(0)
    # # print([len(d) for d in time_duration], seq_lens_list)
    # # 2. Create type list as unstacked list of tensors (same length as time_duration)
    # # type_train = [torch.zeros_like(d, dtype=torch.long) for d in train_duration]
    # # type_test = [torch.zeros_like(d, dtype=torch.long) for d in test_duration]
    
    # # # 3. Pad sequences
    # seq_lens_train = [len(i) for i in time_train]
    # seq_lens_test = [len(i) for i in time_test]
    
    # print("Train after split: ", seq_lens_train)
    # print("Test after split: ", seq_lens_test)
    
    # type_train_seq = [torch.zeros_like(d, dtype=torch.long) for d in time_train]
    # type_test_seq = [torch.zeros_like(d, dtype=torch.long) for d in time_test]
    
    # if seq_len == -1:
    #     time_duration, type_train = utils.padding_full(time_duration, type_train, seq_lens_list, type_size)
    # else:
    #     time_duration, type_train, seq_lens_list = utils.padding_seq_len(time_duration, type_train, type_size, seq_len)
    
    
    # --------------------------
    # Padding
    # --------------------------
    if seq_len == -1:
        # time_train, type_train_seq, seq_lens_train, type_size
        # time_duration, type_train, event_features = utils.padding_full_feats(time_duration, type_train, seq_lens_train, type_size, event_features)
        train_duration, type_train, train_features = padding_full_feats(time_train, type_train_seq, seq_lens_train, type_size, train_features)
        test_duration, type_test, test_features = padding_full_feats(time_test, type_test_seq, seq_lens_test, type_size, test_features)
    else:
        train_duration, type_train, seq_lens_train, train_features = padding_seq_len_feats(time_train, type_train_seq, type_size, seq_len, train_features)
        test_duration, type_test, seq_lens_test, test_features = padding_seq_len_feats(time_test, type_test_seq, type_size, seq_len, test_features)
    
    # Output would have been:
    # print("Before batch: ", train_duration.shape, type_train.shape, seq_lens_train, train_features.shape)
    
    train_data = Data_Batch_Feat(train_duration, type_train, seq_lens_train, train_features)
    train_data = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    
    print("Data Processing Finished...")
    t2 = time.time()
    data_process_time = t2 - t1
    print("Getting data takes: " + str(data_process_time) + " seconds")
    log.write("\n\nGetting data takes: " + str(data_process_time) + " seconds")
    
    print("start training...")
    t3 = time.time()
    if used_model:
        # torch.serialization.add_safe_globals([ConttimeFeatAttnLSTM])
        # torch.serialization.add_safe_globals([nn.Embedding])
        # torch.serialization.add_safe_globals([FeatureAttention])
        # model = ConttimeFeatAttnLSTM(n_types=type_size, lr=lr, extra_dim=20, seq_len=seq_len+1)
        # model.load_state_dict(torch.load("/kaggle/input/neuralhawkes/pytorch/default/1/model_attn_15.pt", weights_only=False))
        model = torch.load("/kaggle/input/neuralhawk/pytorch/default/1/model_attn_15_cont.pt", weights_only=False)
    else:
        model = ConttimeFeatAttnLSTM(n_types=type_size, lr=lr, extra_dim=40, seq_len=seq_len+1)
    
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print("You are using GPU acceleration.")
        log.write("\nYou are using GPU acceleration.")
        print("Number of GPU: ", torch.get_num_threads())
        log.write("\n\nNumber of GPU: " + str((torch.get_num_threads())))
    else:
        device = torch.device("cpu")
        print("CUDA is not Available. You are using CPU only.")
        log.write("\nCUDA is not Available. You are using CPU only.")
        print("Number of cores: ", os.cpu_count())
        log.write("\n\nNumber of cores: " + str(os.cpu_count()))
    
    loss_value = []
    
    log_test_list = []
    log_time_list = []
    log_type_list = []
    rmse_train_list = []
    rmse_test_list = []
    type_accuracy_list = []
    for i in range(num_epochs):
        loss_total = 0
        events_total = 0
        max_len = len(train_data)
        for idx, a_batch in enumerate(train_data):
            durations, type_items, seq_lens, feats = a_batch['duration_seq'], a_batch['event_seq'], a_batch['seq_len'], a_batch['features']
            # The shapes when seqlen=-1: durations:(2, 901), typeitems:(2, 901), seqlens: (2)
            # The shapes when seqlen=10: 
            # print("Batch details:", durations.shape, type_items.shape, seq_lens.shape, feats.shape)
            # print(durations)
            # print(type_items)
            # print(seq_lens)
            # exit(0)
    
            sim_durations, total_time_seqs, time_simulation_index = generate_simulation(durations, seq_lens)
    
            type_items.to(device)
            durations.to(device)
            sim_durations.to(device)
            total_time_seqs.to(device)
            seq_lens.to(device)
            time_simulation_index.to(device)
            feats.to(device)
    
            batch = (type_items, durations, feats)
            loss = model.train_batch(batch, sim_durations, total_time_seqs, seq_lens, time_simulation_index)
            log_likelihood = -loss
            total_size = torch.sum(seq_lens)
            loss_total += log_likelihood.item()
            events_total += total_size.item()
            # print("In epochs {0}, process {1} over {2} is done".format(i, idx, max_len))
        avg_log = loss_total / events_total
        loss_value.append(-avg_log)
        print("The log-likelihood at epochs {0} is {1}".format(i, avg_log))
        log.write("\nThe log likelihood at epochs {0} is {1}".format(i, avg_log))
        print("model saved..")
        torch.save(model, "model.pt")
    
        print("\nvalidating on log likelihood...")
        log_likelihood, type_likelihood, time_likelihood = log_valid(test_duration, type_test, seq_lens_test, test_features, device)
        print("Test log likelihood:", log_likelihood.item())
        log_test_list.append(-log_likelihood.item())
        log_type_list.append(-type_likelihood.item())
        log_time_list.append(-time_likelihood.item())
    
        print("\nvalidating on type prediction accuracy if we know when will next event happens...\n\n")
        accuracy = type_valid(test_duration, seq_lens_test, type_test, test_features)
        type_accuracy_list.append(accuracy)
        print("Accuracy:", accuracy)

        train_rmse = feat_rmse(train_duration, seq_lens_train, train_features, type_train, 10000, model)
        test_rmse = feat_rmse(test_duration, seq_lens_test, test_features, type_test, 10000, model)
        print(f"Train RMSE: {train_rmse}")
        print(f"Test RMSE: {test_rmse}")
        rmse_train_list.append(train_rmse)
        rmse_test_list.append(test_rmse)
    
    train_lle.append(loss_value)
    test_lle.append(log_test_list)
    test_acc.append(type_accuracy_list)
    rmse_train_full.append(rmse_train_list)
    rmse_test_full.append(rmse_test_list)
    
    figure, ax = plt.subplots(nrows=1, ncols=3, figsize=(16, 4))
    ax[0].set_xlabel("epochs")
    ax[0].plot(loss_value, label='training loss')
    ax[0].plot(log_test_list, label='testing loss')
    ax[0].legend()
    ax[1].set_xlabel("epochs")
    ax[1].plot(log_type_list, label='testing type loss')
    ax[1].plot(log_time_list, label='testing time loss')
    ax[1].legend()
    ax[2].set_xlabel("epochs")
    ax[2].set_ylabel('accuracy')
    ax[2].set_title('type-validation-accuracy')
    ax[2].plot(type_accuracy_list, label='dev type accuracy')
    plt.subplots_adjust(top=0.85)
    figure.tight_layout()
    plt.savefig("training.jpg")
    
    t4 = time.time()
    training_time = t4 - t3
    print("training done..")
    print("training takes {0} seconds".format(training_time))
    log.write("\ntraining takes {0} seconds".format(training_time))
    log.close()
    
    print("Saving training loss and validation data...")
    print("If you have a trained model before this, please combine the previous train_date file to" +
        " generate plots that are able to show the whole training information")
    training_info_file = "training-data-" + now + ".txt"
    file = open(training_info_file, 'w')
    file.write("log-likelihood: ")
    file.writelines(str(item) + " " for item in loss_value)
    file.write('\nlog-test-likelihood: ')
    file.writelines(str(item) + " " for item in log_test_list)
    file.write('\nlog-type-likelihood: ')
    file.writelines(str(item) + " " for item in log_type_list)
    file.write('\nlog-time-likelihood: ')
    file.writelines(str(item) + " " for item in log_time_list)
    file.write('\naccuracy: ')
    file.writelines(str(item) + " " for item in type_accuracy_list)
    file.close()
    print("Every works are done!")